In [1]:
# Setup Dash
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64

# Import your CRUD module
from CRUD_Python_Module import CRUD

# Database connection
username = "aacuser"
password = "StrongPassword123"
db = CRUD(username, password)

# Load data
df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=['_id'], inplace=True)

# App
app = JupyterDash(__name__)

# Load image (optional)
# Make sure file exists or comment this out
# encoded_image = base64.b64encode(open("logo.png", 'rb').read())

app.layout = html.Div([

    html.H1("Abimbola Alao Dashboard", style={'textAlign': 'center'}),

    # FILTER OPTIONS
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'All', 'value': 'ALL'},
            {'label': 'Water Rescue', 'value': 'WATER'},
            {'label': 'Mountain/Wilderness', 'value': 'MOUNTAIN'},
            {'label': 'Disaster/Tracking', 'value': 'DISASTER'}
        ],
        value='ALL'
    ),

    html.Hr(),

    # DATA TABLE
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0]
    ),

    html.Br(),

    # CHART + MAP SIDE BY SIDE
    html.Div(style={'display': 'flex'}, children=[

        html.Div(id='graph-id', style={'width': '50%'}),

        html.Div(id='map-id', style={'width': '50%'})

    ])
])

# =========================
# FILTER LOGIC
# =========================
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):

    if filter_type == "WATER":
        query = {"breed": {"$regex": "Labrador|Newfoundland"}}

    elif filter_type == "MOUNTAIN":
        query = {"breed": {"$regex": "German Shepherd|Husky"}}

    elif filter_type == "DISASTER":
        query = {"breed": {"$regex": "Doberman|Belgian Malinois"}}

    else:
        query = {}

    data = db.read(query)
    df = pd.DataFrame.from_records(data)

    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)

    return df.to_dict('records')


# =========================
# CHART (PIE)
# =========================
@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_graphs(viewData):

    if viewData is None:
        return

    dff = pd.DataFrame.from_dict(viewData)

    return dcc.Graph(
        figure=px.pie(dff, names='breed', title='Breed Distribution')
    )


# =========================
# MAP
# =========================
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, selected_rows):

    if viewData is None:
        return

    dff = pd.DataFrame.from_dict(viewData)

    if selected_rows is None or len(selected_rows) == 0:
        row = 0
    else:
        row = selected_rows[0]

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H4("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# ------------------------------
# RUN DASH APP (Codio-compatible)
# ------------------------------
import os
import time
import subprocess

# Generate Codio URL
hostname = subprocess.check_output("hostname", shell=True).decode().strip()

print("\n========================================")
print(" Open your dashboard at:")
print(f" https://{hostname}-8050.codio.io")
print("========================================\n")

# Run Dash app
app.run_server(
    mode='external',   # IMPORTANT: not 'inline'
    debug=False,
    host='0.0.0.0',    # REQUIRED for Codio
    port=8050
)

 * Running on all addresses.
 * Running on http://10.51.105.124:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [19/Apr/2026 10:35:08] "GET /_alive_8cfcbe18-c57c-4b73-b757-34b9fe84850d HTTP/1.1" 200 -



 Open your dashboard at:
 https://sailoravalon-perfumecaviar-8050.codio.io

Dash app running on http://0.0.0.0:8050/
